# Vector search with `sqlitesearch` continued

Reopening the index.

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Search.

In [3]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

Using `sqlitesearch` vector search in RAG.

In [4]:
load_dotenv()
openai_client = OpenAI()

In [5]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [6]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [7]:
vs_index.close()